In [17]:
bronze_path = "Files/bronze/opendata/district_data/district_data.json"

raw_df = (
    spark.read
    .option("multiline", "true")
    .json(bronze_path)
)

raw_df.printSchema()

StatementMeta(, df577cff-934c-42bd-9ef0-9604fcf82188, 19, Finished, Available, Finished, False)

root
 |-- help: string (nullable = true)
 |-- result: struct (nullable = true)
 |    |-- _links: struct (nullable = true)
 |    |    |-- next: string (nullable = true)
 |    |    |-- start: string (nullable = true)
 |    |-- fields: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- id: string (nullable = true)
 |    |    |    |-- type: string (nullable = true)
 |    |-- limit: long (nullable = true)
 |    |-- records: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- AEB: string (nullable = true)
 |    |    |    |-- Codi_Barri: string (nullable = true)
 |    |    |    |-- Codi_Districte: string (nullable = true)
 |    |    |    |-- Data_Referencia: string (nullable = true)
 |    |    |    |-- NACIONALITAT_DOMICILI: string (nullable = true)
 |    |    |    |-- Nom_Barri: string (nullable = true)
 |    |    |    |-- Nom_Districte: string (nullable = true)
 |    |    |    |-- Seccio_Censal: strin

In [18]:
display(
    raw_df.select(
        "success",
        "result.resource_id",
        "result.records"
    )
)

StatementMeta(, df577cff-934c-42bd-9ef0-9604fcf82188, 20, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 556f9eaa-9a38-4edc-b019-0bad7acdcb4b)

In [19]:
from pyspark.sql.functions import explode, col

records_df = (
    raw_df
    .select(
        explode(col("result.records")).alias("record")
    )
)

districts_df = records_df.select("record.*")

display(districts_df)

StatementMeta(, df577cff-934c-42bd-9ef0-9604fcf82188, 21, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4bb94d09-092b-4aa2-b5be-2a0e23e69cdf)

In [20]:
from pyspark.sql.functions import (
    col,
    current_timestamp,
    to_timestamp,
    trim
)

silver_df = (
    districts_df

    # Select and rename columns
    .select(
        col("_id").cast("long").alias("source_id"),
        trim(col("AEB")).alias("aeb"),
        trim(col("Codi_Barri")).alias("neighborhood_code"),
        trim(col("Codi_Districte")).alias("district_code"),
        trim(col("Nom_Barri")).alias("neighborhood_name"),
        trim(col("Nom_Districte")).alias("district_name"),
        trim(col("Seccio_Censal")).alias("census_section"),
        trim(col("NACIONALITAT_DOMICILI")).alias("nationality_code"),
        col("Valor").cast("double").alias("value"),
        to_timestamp(col("Data_Referencia")).alias("reference_date")
    )

    # Remove exact duplicated records
    .dropDuplicates()

    # Processing metadata
    .withColumn("silver_processed_at", current_timestamp())
)

display(silver_df)

StatementMeta(, df577cff-934c-42bd-9ef0-9604fcf82188, 22, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d38d5563-413f-4ec0-bc15-b0546b273728)

In [21]:
from pyspark.sql.functions import count, when

print(f"Bronze records: {districts_df.count()}")
print(f"Silver records: {silver_df.count()}")

display(
    silver_df.select([
        count(
            when(col(c).isNull(), c)
        ).alias(c)
        for c in silver_df.columns
    ])
)

StatementMeta(, df577cff-934c-42bd-9ef0-9604fcf82188, 23, Finished, Available, Finished, False)

Bronze records: 100
Silver records: 100


SynapseWidget(Synapse.DataFrame, 60ce8cc4-4449-4467-bab1-dd733a76b039)

In [22]:
silver_table = "silver_district_context"

(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(silver_table)
)

print(f"Delta table created: {silver_table}")

StatementMeta(, df577cff-934c-42bd-9ef0-9604fcf82188, 24, Finished, Available, Finished, False)

Delta table created: silver_district_context


In [23]:
silver_table_df = spark.table("silver_district_context")

print(f"Rows in Silver Delta table: {silver_table_df.count()}")

silver_table_df.printSchema()

display(silver_table_df)

StatementMeta(, df577cff-934c-42bd-9ef0-9604fcf82188, 25, Finished, Available, Finished, False)

Rows in Silver Delta table: 100
root
 |-- source_id: long (nullable = true)
 |-- aeb: string (nullable = true)
 |-- neighborhood_code: string (nullable = true)
 |-- district_code: string (nullable = true)
 |-- neighborhood_name: string (nullable = true)
 |-- district_name: string (nullable = true)
 |-- census_section: string (nullable = true)
 |-- nationality_code: string (nullable = true)
 |-- value: double (nullable = true)
 |-- reference_date: timestamp (nullable = true)
 |-- silver_processed_at: timestamp (nullable = true)



SynapseWidget(Synapse.DataFrame, 7f04ad58-0423-4cb6-ad7c-829e321d1efb)

In [24]:
from pyspark.sql.functions import col, count, countDistinct, min, max

total_rows = silver_df.count()
unique_ids = silver_df.select("source_id").distinct().count()
duplicate_ids = total_rows - unique_ids
null_critical = silver_df.filter(
    col("source_id").isNull()
    | col("district_code").isNull()
    | col("district_name").isNull()
    | col("value").isNull()
).count()
negative_values = silver_df.filter(col("value") < 0).count()

print(f"Total rows: {total_rows}")
print(f"Unique source_id: {unique_ids}")
print(f"Duplicate source_id: {duplicate_ids}")
print(f"Rows with nulls in critical columns: {null_critical}")
print(f"Rows with negative values: {negative_values}")

display(
    silver_df.select(
        min("value").alias("min_value"),
        max("value").alias("max_value"),
        countDistinct("district_code").alias("district_count"),
        countDistinct("neighborhood_code").alias("neighborhood_count")
    )
)

StatementMeta(, df577cff-934c-42bd-9ef0-9604fcf82188, 26, Finished, Available, Finished, False)

Total rows: 100
Unique source_id: 100
Duplicate source_id: 0
Rows with nulls in critical columns: 0
Rows with negative values: 0


SynapseWidget(Synapse.DataFrame, 0b2c8ee7-b1a8-4dee-a94d-ebf9ffb27402)

In [25]:
assert total_rows > 0, "Silver table is empty"
assert duplicate_ids == 0, "Duplicate source_id values detected"
assert null_critical == 0, "Null values detected in critical columns"
assert negative_values == 0, "Negative values detected in value column"

print("All Silver data quality checks passed.")

StatementMeta(, df577cff-934c-42bd-9ef0-9604fcf82188, 27, Finished, Available, Finished, False)

All Silver data quality checks passed.
